## Preparing Data for a model to forecast the country's total energy consumption

As the title suggests, in this notebook, I intend to focus on preparing data for the model.

I am beginning the load forecasting task immediately after the successful implementation of an offshore wind power forecasting model, as I believe that features derived from the load data could be useful when developing more complex models later on.

In my current approach, I am operating on the premise that the load is, to a large extent, dependent on calendar-based features.

**However, the plan is to try incorporating features such as temperature or storm warnings into the set of predictors, as these can affect heating costs or, theoretically, lead to the suspension of operations and a drop in load.**

In [1]:
# Imports
import pandas as pd
import numpy as np
import os
import pyrootutils
import holidays
from dotenv import load_dotenv

# Needed to load variables from the .env file into the environment
load_dotenv()

# Determining the project root based on the .gitignore file
PROJECT_ROOT = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)
from utils.weather import fetch_load_weather

# Determining data directories
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw_data" / "load_model"
CLEAN_DATA_DIR = PROJECT_ROOT / "data" / "clean_data" / "load_model"
DATASETS_DIR = PROJECT_ROOT / "data" / "datasets" / "load_model"

### In the cell below, I'm creating the basic structure for the future dataset.
For now, I'll use `UTC time` as the index, as it's unique and will be useful for joins.

For the first column, I've used converted `Berlin time`, this will be used to calculate the temporal features for the models.

The step size is 15 minutes, as the target (`load_act`) has that resolution.

In [32]:
# Determining the time interval for the energy load data
START = '2026-01-01'
END = '2026-06-20'

# Creating a DataFrame with UTC and Berlin timestamps
# Index (UTC Timestamps)
utc_index = pd.date_range(
    start=START, 
    end=END, 
    freq='15min', 
    tz='UTC'
)

# Converting UTC timestamps to Berlin time
berlin_time = utc_index.tz_convert('Europe/Berlin')

# Combining both UTC and Berlin timestamps into a single DataFrame
df = pd.DataFrame({
    'dt_berlin': berlin_time
}, index=utc_index)

# Displaying a sample of the DataFrame to verify its structure
df.sample(5)  # Displaying a sample of the DataFrame to verify its structure

,dt_berlin
2026-05-02 20:00:00+00:00,2026-05-02 22:00:00+02:00
2026-02-14 14:00:00+00:00,2026-02-14 15:00:00+01:00
2026-04-06 00:15:00+00:00,2026-04-06 02:15:00+02:00
2026-05-25 18:30:00+00:00,2026-05-25 20:30:00+02:00
2026-04-23 02:45:00+00:00,2026-04-23 04:45:00+02:00


In [35]:
# Encoding the time of day using sine and cosine
local = df['dt_berlin']
minutes_from_midnight = local.dt.hour * 60 + local.dt.minute
time_phase = 2 * np.pi * minutes_from_midnight / 1440

df["time_sin"] = np.sin(time_phase)
df["time_cos"] = np.cos(time_phase)


# Creating a 15-minute day index
quarter_index = minutes_from_midnight // 15
df['quarter_index'] = quarter_index


# Adding a column for the day of the year based on Berlin time
df['doy'] = local.dt.dayofyear

# Encoding the day of the year using sine and cosine
# Calculating the number of days in the year for each date, accounting for leap years
days_in_year = 365 + df["dt_berlin"].dt.is_leap_year.astype(int)
doy_phase = 2 * np.pi * (df["doy"] - 1) / days_in_year

df["doy_sin"] = np.sin(doy_phase)
df["doy_cos"] = np.cos(doy_phase)

# Adding a column for the day of the week based on Berlin time
df['dow'] = local.dt.day_name()


# Adding a column for the holiday weigted index
# Information taken from:
# Monitoringbericht 2025 - Monitoringbericht gemäß § 63 Abs. 3 i. V. m. § 35 EnWG und § 48 Abs. 3 i. V. m. § 53 Abs. 3 GWB
# https://data.bundesnetzagentur.de/Bundesnetzagentur/SharedDocs/Mediathek/Monitoringberichte/MonitoringberichtEnergie2025.pdf
# Pages 96-97
# Consumption volumes (TWh) by states at the VNB level for 2024 (Bundesnetzagentur)
consumption_twh = {
    'NW': 86.94,  # Nordrhein-Westfalen
    'BY': 63.02,  # Bayern
    'BW': 52.09,  # Baden-Württemberg
    'HE': 40.45,  # Hessen
    'NI': 35.58,  # Niedersachsen
    'SN': 19.81,  # Sachsen
    'RP': 17.97,  # Rheinland-Pfalz
    'ST': 13.48,  # Sachsen-Anhalt
    'BE': 12.31,  # Berlin
    'BB': 11.59,  # Brandenburg
    'SH': 11.22,  # Schleswig-Holstein
    'TH': 10.05,  # Thüringen
    'HH': 9.91,   # Hamburg
    'MV': 6.06,   # Mecklenburg-Vorpommern
    'SL': 5.07,   # Saarland
    'HB': 4.47    # Bremen
}

# Normalizing the consumption volumes to get weights for each state
total_twh = sum(consumption_twh.values())
state_weights = {state: val / total_twh for state, val in consumption_twh.items()}


# Creating a column for the weighted holiday indicator based on the state weights and holidays
dates = pd.to_datetime(df['dt_berlin'].dt.date)
years = dates.dt.year.unique().tolist()

# Creating a dictionary of holiday calendars for each state in Germany for the relevant years
state_calendars = {
    state: holidays.Germany(subdiv=state, years=years)
    for state in state_weights
}

# Calculating the weight only for unique dates
unique_dates = dates.dt.date.unique()
date_to_weight = {
    d: sum(weight for state, weight in state_weights.items() if d in state_calendars[state])
    for d in unique_dates
}

# Mapping the calculated weights to the original DataFrame based on the date
df['holiday_index'] = dates.dt.date.map(date_to_weight)

# Creating a boolean mask to identify weekdays (Monday to Friday)
is_weekday = (local.dt.dayofweek < 5)

# Creating a new column 'workday_index' that combines the weekday and holiday information
df["workday_index"] = (
    is_weekday.astype(float)
    * (1.0 - df["holiday_index"])
)

days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Encoding the day of the week into one-hot encoded columns
encoded = pd.get_dummies(
    pd.Categorical(df["dow"], categories=days),
    dtype="int8"
).set_axis([f"is_{day.lower()}" for day in days], axis=1)

# df = pd.concat([df.drop(columns="dow"), encoded.set_axis(df.index)], axis=1)


df.sample(5)

# "quarter_index", "doy", "workday_index"
# These signs are still candidates, they may contain valuable information for boosting, but this requires verification

,dt_berlin,time_sin,time_cos,quarter_index,doy,doy_sin,doy_cos,holiday_index,workday_index,is_monday,is_tuesday,is_wednesday,is_thursday,is_friday,is_saturday,is_sunday,dow
2026-01-18 23:30:00+00:00,2026-01-19 00:30:00+01:00,0.130526,0.991445,2,19,0.304921,0.952378,0.0,1.0,1,0,0,0,0,0,0,Monday
2026-05-14 22:15:00+00:00,2026-05-15 00:15:00+02:00,0.065403,0.997859,1,135,0.741222,-0.671260,0.0,1.0,0,0,0,0,1,0,0,Friday
2026-01-22 19:00:00+00:00,2026-01-22 20:00:00+01:00,-0.866025,0.500000,80,22,0.353676,0.935368,0.0,1.0,0,0,0,1,0,0,0,Thursday
2026-02-12 09:30:00+00:00,2026-02-12 10:30:00+01:00,0.382683,-0.923880,42,43,0.661635,0.749826,0.0,1.0,0,0,0,1,0,0,0,Thursday
2026-01-21 17:30:00+00:00,2026-01-21 18:30:00+01:00,-0.991445,0.130526,74,21,0.337523,0.941317,0.0,1.0,0,0,1,0,0,0,0,Wednesday


In [36]:
df.head(10)

,dt_berlin,time_sin,time_cos,quarter_index,doy,doy_sin,doy_cos,holiday_index,workday_index,is_monday,is_tuesday,is_wednesday,is_thursday,is_friday,is_saturday,is_sunday,dow
2026-01-01 00:00:00+00:00,2026-01-01 01:00:00+01:00,0.258819,0.965926,4,1,0.0,1.0,1.0,0.0,0,0,0,1,0,0,0,Thursday
2026-01-01 00:15:00+00:00,2026-01-01 01:15:00+01:00,0.321439,0.946930,5,1,0.0,1.0,1.0,0.0,0,0,0,1,0,0,0,Thursday
2026-01-01 00:30:00+00:00,2026-01-01 01:30:00+01:00,0.382683,0.923880,6,1,0.0,1.0,1.0,0.0,0,0,0,1,0,0,0,Thursday
2026-01-01 00:45:00+00:00,2026-01-01 01:45:00+01:00,0.442289,0.896873,7,1,0.0,1.0,1.0,0.0,0,0,0,1,0,0,0,Thursday
2026-01-01 01:00:00+00:00,2026-01-01 02:00:00+01:00,0.500000,0.866025,8,1,0.0,1.0,1.0,0.0,0,0,0,1,0,0,0,Thursday
2026-01-01 01:15:00+00:00,2026-01-01 02:15:00+01:00,0.555570,0.831470,9,1,0.0,1.0,1.0,0.0,0,0,0,1,0,0,0,Thursday
2026-01-01 01:30:00+00:00,2026-01-01 02:30:00+01:00,0.608761,0.793353,10,1,0.0,1.0,1.0,0.0,0,0,0,1,0,0,0,Thursday
2026-01-01 01:45:00+00:00,2026-01-01 02:45:00+01:00,0.659346,0.751840,11,1,0.0,1.0,1.0,0.0,0,0,0,1,0,0,0,Thursday
2026-01-01 02:00:00+00:00,2026-01-01 03:00:00+01:00,0.707107,0.707107,12,1,0.0,1.0,1.0,0.0,0,0,0,1,0,0,0,Thursday
2026-01-01 02:15:00+00:00,2026-01-01 03:15:00+01:00,0.751840,0.659346,13,1,0.0,1.0,1.0,0.0,0,0,0,1,0,0,0,Thursday


In [6]:
len(df[df['weighted_holiday'] > 0])/len(df)

0.0644568347527725

In [7]:
state_weights

{'NW': 0.21733913304334784,
 'BY': 0.15754212289385533,
 'BW': 0.13021848907554623,
 'HE': 0.10111994400279987,
 'NI': 0.08894555272236389,
 'SN': 0.04952252387380631,
 'RP': 0.04492275386230688,
 'ST': 0.03369831508424579,
 'BE': 0.030773461326933656,
 'BB': 0.02897355132243388,
 'SH': 0.028048597570121497,
 'TH': 0.025123743812809362,
 'HH': 0.024773761311934404,
 'MV': 0.015149242537873106,
 'SL': 0.012674366281685917,
 'HB': 0.011174441277936103}

### Weather from individual forecast runs

The load profile uses land cells: temperature at 2 m (°C), wind speed at 10 m (m/s), shortwave radiation (W/m², mean over the preceding hour), and cloud cover (%). The grid targets about 20 locations; it is spatially uniform, not consumption-weighted. Freeze the coordinates alongside the raw data before comparing models.

For each Berlin delivery day, use the previous day's 00:00 UTC run. The fetcher retains one buffer hour on each side, including DST days. Keep `datetime_run_utc` when joining or interpolating: adjacent runs deliberately overlap in their buffer hours. The returned data is hourly and has not yet been joined to the quarter-hour load grid.

In [8]:
from utils.germany_grid_map import generate_germany_grid, visualize_grid

# Use the repository's boundary snapshot for reproducible coordinates.
load_points = generate_germany_grid(
    n_points=20,
    states_url=(PROJECT_ROOT / 'utils/germany_states_low.geojson').as_uri(),
)
load_points

# visualize_grid(load_points)

[germany_grid] Loading federal state boundaries...
[germany_grid] Finding grid spacing for about 20 points...
[germany_grid] Grid spacing: 140.27 km
[germany_grid] Points inside Germany: 20
[germany_grid] Assigning points to federal states...
[germany_grid] Done. Total points: 20


{1: [8.935644, 48.358116, 'Baden-Württemberg'],
 2: [10.82771, 48.360139, 'Bayern'],
 3: [12.718808, 48.329824, 'Bayern'],
 4: [7.945168, 49.43719, 'Rheinland-Pfalz'],
 5: [9.879022, 49.455853, 'Baden-Württemberg'],
 6: [11.813026, 49.44134, 'Bayern'],
 7: [6.90947, 50.506619, 'Rheinland-Pfalz'],
 8: [8.886832, 50.542796, 'Hessen'],
 9: [10.865672, 50.544926, 'Thüringen'],
 10: [12.843362, 50.513005, 'Sachsen'],
 11: [7.84762, 51.620384, 'Nordrhein-Westfalen'],
 12: [9.873275, 51.640069, 'Niedersachsen'],
 13: [11.899109, 51.62476, 'Sachsen-Anhalt'],
 14: [13.922253, 51.574493, 'Brandenburg'],
 15: [8.832033, 52.72618, 'Niedersachsen'],
 16: [10.90829, 52.728431, 'Sachsen-Anhalt'],
 17: [12.98317, 52.694702, 'Brandenburg'],
 18: [9.866803, 53.823428, 'Schleswig-Holstein'],
 19: [11.996039, 53.80722, 'Mecklenburg-Vorpommern'],
 20: [8.770126, 54.909071, 'Schleswig-Holstein']}

In [ ]:
START = '2026-06-25'
END = '2026-11-17'

# START/END below are interpreted as inclusive Berlin delivery dates for weather.
WEATHER_RUN_START = (pd.Timestamp(START) - pd.DateOffset(days=1)).strftime('%Y-%m-%d')
WEATHER_RUN_END = (pd.Timestamp(END) - pd.DateOffset(days=1)).strftime('%Y-%m-%d')
FETCH_LOAD_WEATHER = True  # Enable explicitly for the historical download.

# 20 coordinates with a 3-second interval fit within the minute API-limits
if FETCH_LOAD_WEATHER:
    import importlib
    from utils import weather as weather_module

    weather_module = importlib.reload(weather_module)
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

    temp_checkpoint = RAW_DATA_DIR / '_tmp_load_ECMWF_singleruns.parquet'
    print(f'Weather module: {weather_module.__file__}')
    print(f'Temp checkpoint: {temp_checkpoint.resolve()}')

    load_weather = weather_module.fetch_load_weather(
        WEATHER_RUN_START,
        WEATHER_RUN_END,
        points=load_points,
        checkpoint_path=temp_checkpoint,
    )

    dt_utc = pd.to_datetime(load_weather['datetime_utc'], utc=True)
    actual_start = dt_utc.min().strftime('%Y-%m-%d')
    actual_end = dt_utc.max().strftime('%Y-%m-%d')

    stem = f'load_ECMWF_singleruns_{actual_start}_to_{actual_end}'
    final_parquet = RAW_DATA_DIR / f'{stem}.parquet'

    load_weather.to_parquet(final_parquet, index=False)
    if temp_checkpoint.exists() and temp_checkpoint != final_parquet:
        temp_checkpoint.unlink()

    pd.DataFrame.from_dict(
        load_points, orient='index', columns=['longitude', 'latitude', 'region']
    ).rename_axis('point_id').to_json(RAW_DATA_DIR / f'{stem}_points.json', orient='table')

    print(f'Saved parquet: {final_parquet.resolve()}')

Weather module: D:\python_projects\entsoe_2nd_try\utils\weather.py
Temp checkpoint: D:\python_projects\entsoe_2nd_try\data\raw_data\load_model\_tmp_load_ECMWF_singleruns.parquet
Run range 2026-06-24 -> 2026-11-16 (ECMWF IFS HRES)
  Target: calendar day D+1 in Europe/Berlin, +/-1-hour buffer
  Request pacing: 3s after each run for 20 points
  [Single Run | ecmwf_ifs] run 2026-06-24T00:00, 20 points...
  [Single Run | ecmwf_ifs] run 2026-06-25T00:00, 20 points...
  [Single Run | ecmwf_ifs] run 2026-06-26T00:00, 20 points...
  [Single Run | ecmwf_ifs] run 2026-06-27T00:00, 20 points...
  [Single Run | ecmwf_ifs] run 2026-06-28T00:00, 20 points...
  [Single Run | ecmwf_ifs] run 2026-06-29T00:00, 20 points...
  [Single Run | ecmwf_ifs] run 2026-06-30T00:00, 20 points...
  [Single Run | ecmwf_ifs] run 2026-07-01T00:00, 20 points...
  [Single Run | ecmwf_ifs] run 2026-07-02T00:00, 20 points...
  [Single Run | ecmwf_ifs] run 2026-07-03T00:00, 20 points...
  [Single Run | ecmwf_ifs] run 2026-07

In [10]:
load_weather.sample(10)

,datetime_run_utc,datetime_utc,datetime_berlin,point_id,region,latitude,longitude,temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover
21480,2024-05-29 00:00:00+00:00,2024-05-30 02:00:00+00:00,2024-05-30 04:00:00+02:00,4,Rheinland-Pfalz,49.455181,7.900172,12.450000,3.571064,0.0,100.0
56419,2024-08-31 00:00:00+00:00,2024-08-31 21:00:00+00:00,2024-08-31 23:00:00+02:00,9,Thüringen,50.579964,10.831859,16.000000,4.063558,0.0,26.0
69141,2024-08-20 00:00:00+00:00,2024-08-21 05:00:00+00:00,2024-08-21 07:00:00+02:00,11,Nordrhein-Westfalen,51.634445,7.854545,14.800000,4.128256,9.0,0.0
19550,2024-03-15 00:00:00+00:00,2024-03-16 22:00:00+00:00,2024-03-16 23:00:00+01:00,4,Rheinland-Pfalz,49.455181,7.900172,5.800000,0.269258,0.0,97.0
17247,2024-08-24 00:00:00+00:00,2024-08-25 07:00:00+00:00,2024-08-25 09:00:00+02:00,3,Bayern,48.330402,12.663317,20.700001,3.153173,170.0,100.0
7555,2024-04-23 00:00:00+00:00,2024-04-24 13:00:00+00:00,2024-04-24 15:00:00+02:00,2,Bayern,48.330402,10.854271,5.750000,5.277309,531.0,91.0
38847,2024-11-13 00:00:00+00:00,2024-11-14 01:00:00+00:00,2024-11-14 02:00:00+01:00,6,Bayern,49.455181,11.772805,-0.900000,1.649242,0.0,33.0
25895,2024-11-14 00:00:00+00:00,2024-11-15 23:00:00+00:00,2024-11-16 00:00:00+01:00,4,Rheinland-Pfalz,49.455181,7.900172,3.900000,1.110180,0.0,53.0
114320,2024-08-07 00:00:00+00:00,2024-08-08 22:00:00+00:00,2024-08-09 00:00:00+02:00,18,Schleswig-Holstein,53.813705,9.884393,16.799999,2.673013,0.0,100.0
46825,2024-05-04 00:00:00+00:00,2024-05-04 21:00:00+00:00,2024-05-04 23:00:00+02:00,8,Hessen,50.509663,8.904593,13.500000,3.277575,0.0,100.0


In [19]:
pd.read_parquet( RAW_DATA_DIR / 'load_ECMWF_singleruns_2025-07-24_to_2026-06-23.parquet').describe()

,point_id,latitude,longitude,temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover
count,174200.000000,174200.000000,174200.000000,174200.000000,174200.000000,174200.000000,174200.000000
mean,10.500000,51.131808,10.411189,8.847575,3.241398,122.281151,64.447906
std,5.766298,1.880977,1.916167,7.597214,1.756270,202.819977,41.410919
min,1.000000,48.330402,6.837456,-15.700000,0.000000,0.000000,0.000000
25%,5.750000,49.455181,8.862703,3.100000,1.930026,0.000000,17.000000
50%,10.500000,51.072054,10.372900,8.700000,3.000833,0.000000,93.000000
75%,15.250000,52.688927,11.950420,14.150000,4.288283,175.000000,100.000000
max,20.000000,54.938488,13.883847,35.150002,12.701180,945.000000,100.000000


In [22]:
import json
from pathlib import Path

base_dir = RAW_DATA_DIR  # путь к нужной директории

files = [
    base_dir / "load_ECMWF_singleruns_2026-06-24_to_2026-11-17_points.json",
    base_dir / "load_ECMWF_singleruns_2024-11-17_to_2025-07-25_points.json",
    base_dir / "load_ECMWF_singleruns_2024-03-14_to_2024-11-19_points.json",
]

loaded = [json.loads(p.read_text(encoding="utf-8")) for p in files]

print(
    "1 и 2 идентичны:", loaded[0] == loaded[1]
)  #[cite: 1]
print(
    "1 и 3 идентичны:", loaded[0] == loaded[2]
)  #[cite: 1]
print(
    "2 и 3 идентичны:", loaded[1] == loaded[2]
)  #[cite: 1]

1 и 2 идентичны: True
1 и 3 идентичны: True
2 и 3 идентичны: True
